# Worker del analizador de chats en Google Colab

1. (Opcional) *Entorno de ejecución → Cambiar tipo → GPU T4*.
2. *Entorno de ejecución → Ejecutar todas*.
3. Cuando lo pida, pegá la **URL de CloudAMQP** (`amqps://…`) y el **WORKER_TOKEN** de `distribuido/config.env`. También se pueden guardar en los *Secretos* de Colab (ícono de llave) con esos mismos nombres, y entonces no pregunta.

El texto se clasifica con **Ollama** (qwen2.5:3b), igual que en la laptop; la celda 2b lo instala (la primera vez tarda unos 2 minutos).

La última celda se queda corriendo: ese es el worker. La dirección para descargar fotos y audios **no hace falta**: el coordinador la manda en cada tarea (la de su túnel de Cloudflare).

⚠️ No guardar este notebook en GitHub con las credenciales escritas. Procesar con estos workers **solo el dataset generado**, nunca chats reales.

In [ ]:
# 1. Código del proyecto
import os
REPO = "https://github.com/SharonBlanco/Analizador-de-Chats-de-Whatsapp.git"
if not os.path.isdir("/content/proyecto"):
    !git clone -q {REPO} /content/proyecto
else:
    !git -C /content/proyecto pull -q
%cd /content/proyecto
!nvidia-smi --query-gpu=name,memory.total --format=csv 2>/dev/null || echo "Sin GPU: el worker usará la CPU" 

In [ ]:
# 2. Librerías (Colab ya trae PyTorch con CUDA y ffmpeg)
!pip install -q pika==1.3.2 pysentimiento ultralytics faster-whisper

In [ ]:
# 2b. Ollama: el mismo detector de texto que la laptop (qwen2.5:3b), así todos clasifican igual
import subprocess, time, urllib.request
!apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(30):                     # esperar a que el servidor responda
    try:
        urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2); break
    except OSError:
        time.sleep(1)
!ollama pull qwen2.5:3b 2>&1 | tail -1
os.environ["WA_TEXTO_MOTOR"] = "ollama"     # si Ollama falla, la detección de texto lo indica (no usa otro motor)
!ollama run qwen2.5:3b "Respondé solo: listo" 2>/dev/null


In [ ]:
# 3. Credenciales: de los Secretos de Colab o pegadas al correr (no quedan escritas en el notebook)
from getpass import getpass
def dato(clave, pregunta):
    try:
        from google.colab import userdata
        v = userdata.get(clave)
        if v:
            return v
    except Exception:
        pass
    return getpass(pregunta)

os.environ["RABBIT_URL"] = dato("RABBIT_URL", "URL de CloudAMQP (amqps://…): ")
os.environ["WORKER_TOKEN"] = dato("WORKER_TOKEN", "WORKER_TOKEN: ")

NOMBRE = "colab-1"                     # cambiar a colab-2 en el segundo worker
os.environ.update(WORKER_ID=NOMBRE, HOST_NAME=NOMBRE, PYTHONUNBUFFERED="1")
print("Worker", NOMBRE, "listo: atiende texto, fotos, audios y adjuntos")

In [ ]:
# 4. Worker (se queda corriendo). El supervisor lo reinicia si se queda sin memoria (M1).
!python distribuido/worker/supervisor.py